In [141]:
# load env variables 
from dotenv import load_dotenv
load_dotenv()

True

In [142]:
# create the loader 
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("paper.pdf")
type(loader)

# actually read the pdf
docs=loader.load()  
type(docs),len(docs),type(docs[0]),docs[0].metadata

(list,
 15,
 langchain_core.documents.base.Document,
 {'producer': 'pdfTeX-1.40.25',
  'creator': 'LaTeX with hyperref',
  'creationdate': '2024-04-10T21:11:43+00:00',
  'source': 'paper.pdf',
  'file_path': 'paper.pdf',
  'total_pages': 15,
  'format': 'PDF 1.5',
  'title': '',
  'author': '',
  'subject': '',
  'keywords': '',
  'moddate': '2024-04-10T21:11:43+00:00',
  'trapped': '',
  'modDate': 'D:20240410211143Z',
  'creationDate': 'D:20240410211143Z',
  'page': 0})

In [143]:
# create splitter 
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
type(splitter)

#actually split
chunks=splitter.split_documents(docs) 
print(type(chunks)),print(len(chunks)),print(chunks[0].metadata)

<class 'list'>
94
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': 'paper.pdf', 'file_path': 'paper.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}


(None, None, None)

In [144]:
# create embedding model
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# test the embedding model 
vector=embeddings.embed_query("what is transformer")
print(type(vector))

print(vector)
print(len(vector))

#will print dimension of vector or no of words in a sentence
# dimension or no of words depends on embedding model used 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3034.50it/s]


<class 'list'>
[-0.1854417473077774, 0.04515491798520088, -0.026022404432296753, -0.0036607314832508564, 0.022438116371631622, -0.016202835366129875, -0.04091993346810341, 0.07431329786777496, 0.052100397646427155, -0.015700768679380417, -0.015121008269488811, 0.04672654718160629, 0.030482813715934753, 0.044781673699617386, -0.020520301535725594, 0.01667211577296257, -0.05998294800519943, 0.020185714587569237, -0.07053320109844208, -0.09665405005216599, -0.03314298018813133, 0.07608352601528168, -0.06771989166736603, -0.02875690907239914, 0.04429774358868599, -0.013640015386044979, 0.031212197616696358, -0.0586247444152832, -0.05167585238814354, -0.021882962435483932, -0.02064366452395916, -0.041383977979421616, -0.09227176755666733, 0.07309770584106445, -0.10471798479557037, 0.055586062371730804, -0.010495931841433048, -0.01680683344602585, 0.02322673238813877, 0.013393099419772625, 0.0379340685904026, -0.08773507177829742, 0.020271698012948036, -0.057363998144865036, -0.0447370111942

In [145]:
# create chromadb
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print(type(vectorstore))

<class 'langchain_chroma.vectorstores.Chroma'>


In [146]:
# create the retriever - another object that knows how to search.
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4} # gives top 4 relevant chunks
)

#search and find nearest vectors
retrieved_docs = retriever.invoke("what is transformer?")
len(retrieved_docs)

4

In [147]:
# inspect each chunk
for i,doc in enumerate(retrieved_docs):
    print(f"Chunk {i+1}") # or ("chunk ",i+1)
    print(doc.metadata)

Chunk 1
{'source': 'paper.pdf', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.25', 'creationDate': 'D:20240410211143Z', 'author': '', 'modDate': 'D:20240410211143Z', 'keywords': '', 'page': 2, 'file_path': 'paper.pdf', 'total_pages': 15, 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'format': 'PDF 1.5', 'creationdate': '2024-04-10T21:11:43+00:00', 'subject': '', 'title': ''}
Chunk 2
{'producer': 'pdfTeX-1.40.25', 'creationdate': '2024-04-10T21:11:43+00:00', 'modDate': 'D:20240410211143Z', 'trapped': '', 'keywords': '', 'creationDate': 'D:20240410211143Z', 'subject': '', 'author': '', 'page': 2, 'total_pages': 15, 'moddate': '2024-04-10T21:11:43+00:00', 'format': 'PDF 1.5', 'file_path': 'paper.pdf', 'source': 'paper.pdf', 'creator': 'LaTeX with hyperref', 'title': ''}
Chunk 3
{'file_path': 'paper.pdf', 'title': '', 'creationdate': '2024-04-10T21:11:43+00:00', 'creator': 'LaTeX with hyperref', 'source': 'paper.pdf', 'modDate': 'D:20240410211143Z', 'keywords': '', 't

In [148]:
# create llm
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.1-8b-instant",
    temperature=0,
)

In [149]:
# create prompt template
from langchain_core.prompts import PromptTemplate

prompt= PromptTemplate(
    template="""
    You are an ai research assistant.
    answer only from provided context
    if answer is not present, reply:
    "i don't know."

    Context:
    {context}

    Question:
    {question}

    Answer:
""",
    input_variables=["context","question"]
)

In [150]:
# create str output parser 
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [151]:
# convert documents to text
# retriever returns a list of Document objects, but the prompt needs plain text.
def format_docs(docs):
    return "\n\n".join( # join with 2 line break in between 
        doc.page_content for doc in docs
    )

In [152]:
# build a rag chain
from langchain_core.runnables import RunnablePassthrough

chain=( 
    {
        "context": retriever | format_docs,
        "question":RunnablePassthrough()
    } | prompt | llm | parser
)

In [153]:
# ask a question 
response = chain.invoke(
    "What is transformer ?"
)

print(response)

The Transformer is a model architecture, as shown in Figure 1, that follows an overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.


## features 

In [154]:
#paper overview 
first_page= docs[0].page_content

overview_prompt=PromptTemplate(
    template="""
You are an AI Research Assistant.

Extract the following information from the research paper.

Return only the information.

Title:

Authors:

Conference:

Year:

Research Domain:

One-line Description:

Paper:

{paper}
""",
input_variables=["paper"]
)

# we break a lot of lines bw sentences to 
# make the output structured and better code readability

overview_chain =  overview_prompt | llm | parser

response= overview_chain.invoke(
    {
        "paper":first_page
    }
)
print(response)

- Title: Attention Is All You Need
- Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin
- Conference: 31st Conference on Neural Information Processing Systems (NIPS 2017)
- Year: 2017
- Research Domain: Natural Language Processing
- One-line Description: A new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.


In [155]:
# paper metadata
# structured extraction - json schema for fixed format of output
from pydantic import BaseModel, Field

class PaperMetadata(BaseModel):
    title: str = Field(description="Title of the research paper")
    dataset: str = Field(description="Dataset used")
    architecture: str = Field(description="Model architecture")
    optimizer: str = Field(description="Optimizer used")
    epochs: str = Field(description="Number of training epochs")
    learning_rate: str = Field(description="Learning rate")
    batch_size: str = Field(description="Batch size")
    metrics: str = Field(description="Evaluation metrics")
# basemodel for feature validation 
# field description for llm context 

In [156]:
# reusable extract prompt
extract_prompt=PromptTemplate(
    template="""
You are an AI Research Assistant.

Answer ONLY from the given context.

If the answer is not present, return exactly:
"Not specified"

Context:
{context}

Question:
{question}

Answer:""",
    input_variables=["context", "question"]
)

extract_chain= extract_prompt | llm | parser

In [157]:
#config
FIELDS={
    "title": "What is the title of the paper?",
    "dataset": "What dataset was used?",
    "architecture": "What model architecture was proposed?",
    "optimizer": "What optimizer was used?",
    "epochs": "How many training epochs were used?",
    "learning_rate": "What learning rate was used?",
    "batch_size": "What batch size was used?",
    "metrics": "What evaluation metrics were reported?"
}

In [158]:
# page number 
def get_sources(docs):
    pages= []
    for doc in docs:
        pages.append(doc.metadata["page"]+1)
    return sorted(set(pages))

In [159]:
def extract(question):
    docs= retriever.invoke(question) #retrieve relevant chunks
    context = format_docs(docs)    # Convert documents to plain text
    answer=extract_chain.invoke({
        "context": context,
        "question": question
    })
    sources= get_sources(docs)
    return{
        "answer":answer.strip(),
        "sources":sources
    }

In [160]:
results = {}
citations = {}

for field, question in FIELDS.items():

    result = extract(question)

    results[field] = result["answer"]

    citations[field] = result["sources"]

In [161]:
# paper summary
summary_prompt = PromptTemplate(
    template="""
You are an AI Research Assistant.

Using ONLY the provided context,

Generate:

1. A short summary (4–5 sentences)

2. Five key bullet points

3. Main contribution

Context:

{context}
""",
    input_variables=["context"]
)

summary_chain = summary_prompt | llm | parser

def summarize():
    docs=retriever.invoke(
        "Summarize this research paper."
    )
    context= format_docs(docs)
    return summary_chain.invoke(
        {
            "context":context
        }
    )

summary = summarize()

print(summary)

**Summary (4-5 sentences)**

The provided context appears to be the beginning of a research paper titled "Attention Is All You Need." The paper is written by a team of researchers from Google Brain and Google Research, including Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, and Aidan N. Gomez. The paper grants permission to reproduce tables and figures for journalistic or scholarly works, provided proper attribution is given. However, the actual content of the paper is not provided in the given context. The title suggests that the paper may discuss the use of attention mechanisms in deep learning models.

**Five Key Bullet Points**

- The paper is titled "Attention Is All You Need" and is written by researchers from Google Brain and Google Research.
- The paper grants permission to reproduce tables and figures for journalistic or scholarly works, provided proper attribution is given.
- The authors of the paper are Ashish Vaswani, Noam Shazeer, Niki Parmar, Ja

In [162]:
# implementation checklist
checklist_prompt = PromptTemplate(
    template="""
You are an expert Machine Learning Engineer.

Using ONLY the provided context,

Generate a step-by-step implementation checklist for reproducing this research paper.

Rules:

- Use numbered steps.
- Mention dataset.
- Mention preprocessing if available.
- Mention architecture.
- Mention optimizer.
- Mention learning rate.
- Mention batch size.
- Mention epochs.
- Mention evaluation metric.
- If something is missing, write:
"Not specified."

Context:
{context}
""",
    input_variables=["context"]
)

checklist_chain = checklist_prompt | llm | parser

def generate_checklist():
    docs=retriever.invoke(
        "How can this research paper be reproduced."
    )
    context= format_docs(docs)
    return checklist_chain.invoke(
        {
            "context":context
        }
    )

checklist = generate_checklist()

print(checklist)

Based on the provided context, it appears that the research paper is "Attention Is All You Need" by Ashish Vaswani et al. However, the provided text does not contain any information about the dataset, preprocessing, architecture, optimizer, learning rate, batch size, epochs, or evaluation metric used in the paper. Therefore, I will provide a checklist with "Not specified" for the missing information.

Here is the step-by-step implementation checklist for reproducing the research paper:

1. **Dataset**: Not specified.
2. **Data Preprocessing**: Not specified.
3. **Data Loading**: Load the dataset into a suitable format for training and testing.
4. **Model Architecture**: Not specified.
5. **Model Implementation**: Implement the model architecture using a deep learning framework such as TensorFlow or PyTorch.
6. **Optimizer**: Not specified.
7. **Learning Rate**: Not specified.
8. **Batch Size**: Not specified.
9. **Epochs**: Not specified.
10. **Evaluation Metric**: Not specified.
11. *

In [163]:
planner_prompt = PromptTemplate(
    template="""
You are an expert Machine Learning Engineer.

Below is the extracted metadata from a research paper.

Metadata:
{metadata}

Generate:

1. Step-by-step implementation plan.
2. Required libraries.
3. Suggested project folder structure.
4. Training workflow.
5. Evaluation workflow.

If any information is missing, clearly mention it.

""",
    input_variables=["metadata"]
)

planner_chain = planner_prompt | llm | parser

def generate_plan(metadata):

    return planner_chain.invoke(
        {
            "metadata":metadata.model_dump_json(indent=2) # easier for llm to understand
        }
    )

plan = generate_plan(metadata)
print(plan)

**Implementation Plan**

Based on the provided metadata, we can infer that the research paper is related to a transformer-based model, likely a variant of the BERT or similar architecture. However, some crucial information is missing, such as the dataset, epochs, batch size, and the specific model architecture. We will assume some default values for these parameters.

**Required Libraries**

To implement the model, we will need the following libraries:

1. **TensorFlow** or **PyTorch**: For building and training the model.
2. **Transformers**: For using pre-trained transformer models and their associated libraries.
3. **Numpy**: For numerical computations.
4. **Pandas**: For data manipulation and loading.
5. **Scikit-learn**: For evaluation metrics (BLEU score).

**Suggested Project Folder Structure**

```markdown
project/
|---- data/
|       |---- dataset.csv (or any other dataset format)
|---- models/
|       |---- transformer_model.py
|---- utils/
|       |---- data_utils.py
|      

In [164]:
plan = generate_plan(metadata)

print(plan)

**Implementation Plan**

Based on the provided metadata, we can infer that the research paper is related to a transformer-based model, likely a variant of the BERT or similar architecture. However, some crucial information is missing, such as the dataset, epochs, batch size, and the specific model architecture. We will assume some default values for these parameters.

**Required Libraries**

To implement the model, we will need the following libraries:

1. **TensorFlow** or **PyTorch**: For building and training the model.
2. **Transformers**: For using pre-trained transformer models and their associated libraries.
3. **Numpy**: For numerical computations.
4. **Pandas**: For data manipulation and loading.
5. **Scikit-learn**: For evaluation metrics (BLEU score).

**Suggested Project Folder Structure**

```markdown
project/
|---- data/
|       |---- dataset.csv (or any other dataset format)
|---- models/
|       |---- transformer_model.py
|---- utils/
|       |---- data_utils.py
|      

In [165]:
# research report builder 
research_report = {
    "overview": response,
    "metadata": metadata.model_dump(),
    "citations": citations,
    "implementation_plan": plan
}

research_report.keys()

metadata = PaperMetadata(**results)
#The ** operator unpacks a dictionary into keyword arguments.
print(metadata)

title='Not specified' dataset='Not specified' architecture='Scaled dot-product attention, multi-head attention and the parameter-free position representation.' optimizer='The Adam optimizer [20] with β1 = 0.9, β2 = 0.98 and ϵ = 10−9.' epochs='Not specified' learning_rate='lrate = d−0.5\nmodel · min(step_num−0.5, step_num · warmup_steps−1.5)\n\nThis corresponds to a formula that varies the learning rate over the course of training.' batch_size='Not specified' metrics='BLEU'


In [166]:
risk_prompt = PromptTemplate(
    template="""
You are an ML Research Engineer.

Below is metadata extracted from a research paper.

Metadata:

{metadata}

Analyse:

1. Missing information.
2. Reproduction difficulty.
3. Possible risks.
4. Suggestions before implementation.

Return a structured report.
""",
    input_variables=["metadata"]
)

risk_chain = risk_prompt | llm | parser

def analyze_risk(metadata):
    return risk_chain.invoke(
        {
            "metadata": metadata.model_dump_json(indent=2)
        }
    )

risk_report = analyze_risk(metadata)
print(risk_report)

**Structured Report: Analysis of Research Paper Metadata**

**I. Missing Information**

1. **Dataset**: The dataset used for training and evaluation is not specified, which makes it difficult to reproduce the results.
2. **Architecture**: Although the architecture is mentioned, the specific model implementation (e.g., number of layers, hidden dimensions) is not provided.
3. **Epochs**: The number of training epochs is not specified, which can affect the model's performance and convergence.
4. **Batch Size**: The batch size used for training is not specified, which can impact the model's training speed and convergence.
5. **Title**: The title of the research paper is not specified, which can make it difficult to identify the paper and its content.

**II. Reproduction Difficulty**

1. **Lack of Specificity**: The metadata lacks specific details about the model implementation, training parameters, and evaluation metrics, making it challenging to reproduce the results.
2. **Unclear Learnin

In [168]:
summary_prompt = PromptTemplate(
    template="""
You are an AI Research Assistant.

Based on the following information

Overview:

{overview}

Metadata:

{metadata}

Implementation Plan:

{plan}

Risk Report:

{risk}

Write an executive summary.

Maximum 250 words.
""",
input_variables=[
"overview",
"metadata",
"plan",
"risk"
]
)

summary_chain = summary_prompt | llm | parser

def executive_summary(
    overview,
    metadata,
    plan,
    risk
):

    return summary_chain.invoke(
        {
            "overview": overview,
            "metadata": metadata.model_dump_json(indent=2),
            "plan": plan,
            "risk": risk
        }
    )

summary = executive_summary(
    response,
    metadata,
    plan,
    risk_report
)

print(summary)

**Executive Summary:**

The research paper "Attention Is All You Need" by Ashish Vaswani et al. (2017) proposes a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. The paper's metadata lacks specific details about the model implementation, training parameters, and evaluation metrics, making it challenging to reproduce the results.

**Key Risks:**

1. **Inconsistent Results**: Lack of specific details about the model implementation and training parameters can lead to inconsistent results when attempting to reproduce the paper's findings.
2. **Overfitting**: The model's performance may suffer from overfitting due to the lack of information about the training parameters and evaluation metrics.
3. **Security Risks**: The use of a custom learning rate formula and the Adam optimizer with specific hyperparameters may introduce security risks if not properly validated.

**Recommendations:**

1. **Provid